# Using the autograder

## Grading one response

This portion of the documentation will detail how to use the autograder, and perform iterative tasks. For example, we have an exam question that is as follows:

**What does a p-value of 0.03 indicate in a hypothesis test assuming alpha = 0.05?**

With the following rubric components:
1. States that p-value is less than 0.05 (+1 point);
2. States that the null hypothesis is rejected (+1 point)

First, we import the Autograder class. (Ignore the error below)

In [1]:
from backend.call_llm import Autograder

ModuleNotFoundError: No module named 'backend'

We then initialize the autograder by passing it what model we want it to instantiate.

In [8]:
autograder = Autograder(llm_model="llama3.1:8b")

2025-03-31 14:44:28,052 - backend.call_llm - INFO - Logging enabled
2025-03-31 14:44:28,053 - backend.call_llm - INFO - Initializing autograder...


We then load in the rubric components by using the `set_rubric()` method. Before we do so, we have to organize it as a **list of tuples**. These tuples should contain `(question (str), score (int))`. So our first rubric component would be `("states that p-value is less than 0.05", 1)`, meaning this rubric component is worth 1 point.

Lets organize and store the rubric components into a list of tuples.

In [9]:
rubric_components = [
    ("States that p-value is less than 0.05", 1),
    ("States that the null hypothesis is rejected", 1),
]

type(rubric_components)

list

Now that this is organized, we can feed it into the autograder using the `set_rubric()` method.

In [10]:
autograder.set_rubric(rubric_components)

2025-03-31 14:44:56,485 - backend.call_llm - INFO - Parsing rubric components...
2025-03-31 14:44:56,487 - backend.call_llm - INFO - 2 rubric component(s) set.


Now that the autograder knows what to base the evaluation off of, we can ask it to grade a student response. Here is an example student response:

**Since p is less than alpha, the null is rejected.**

We now use the `evaluate()` method to evaluate this response.

Since this function returns an evaluation, we must store it in a variable first.

In [13]:
evaluation = autograder.evaluate(response="Since p is less than alpha, the null is rejected.")

print(evaluation)

2025-03-31 14:45:33,485 - backend.call_llm - INFO - Initiating request to Ollama.
2025-03-31 14:45:33,692 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/generate "HTTP/1.1 200 OK"


## CRITERION: States that p-value is less than 0.05
EXPLANATION: The student's response states "Since p is less than alpha", which implies that the p-value is indeed less than 0.05 (alpha). This aligns with the rubric requirement.
SCORE: 1/1

## CRITERION: States that the null hypothesis is rejected
EXPLANATION: The student's response explicitly states, "the null is rejected", directly matching the required outcome stated in the rubric.
SCORE: 1/1

## TOTAL_SCORE: 2/2


Great!

The autograder assigned it 2/2, which is what we expect because the student response implied that p-value was less than 0.05 (less than alpha), and states directly that the null is rejected.

## Grading multiple responses

At the moment, batch grading can be implemented using for loops. We are working on more in-built features to optimize batch grading.

We first organize student responses into a list.

In [19]:
student_responses = [
    "A p-value of 0.03 means that the alternative hypothesis is rejected, and the null is accepted.",
    "Since p is less than alpha, the null is rejected.",
]

We then initialize an instance of autograder and set up the rubric.

In [20]:
autograder = Autograder(llm_model="llama3.1:8b")

autograder.set_rubric([
    ("States that p-value is less than 0.05", 1),
    ("States that the null hypothesis is rejected", 1),
])

2025-03-31 14:55:49,946 - backend.call_llm - INFO - Logging enabled
2025-03-31 14:55:49,948 - backend.call_llm - INFO - Initializing autograder...
2025-03-31 14:55:49,969 - backend.call_llm - INFO - Parsing rubric components...
2025-03-31 14:55:49,969 - backend.call_llm - INFO - 2 rubric component(s) set.


Since we want to capture the evaluations, we initialize an empty list to store the evaluations as they are completed.

In [21]:
evaluations = []

We can now set up a for loop to iterate through the student responses to grade these student responses.

In [22]:
for i, response in enumerate(student_responses):
    data = {'id': i, 'evaluation': autograder.evaluate(response)}
    evaluations.append(data)

2025-03-31 14:55:51,406 - backend.call_llm - INFO - Initiating request to Ollama.
2025-03-31 14:55:52,368 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/generate "HTTP/1.1 200 OK"
2025-03-31 14:55:57,955 - backend.call_llm - INFO - Initiating request to Ollama.
2025-03-31 14:55:58,392 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/generate "HTTP/1.1 200 OK"


In [23]:
print(evaluations)

[{'id': 0, 'evaluation': "## CRITERION: States that p-value is less than 0.05\nEXPLANATION: The student's response incorrectly states that a p-value of 0.03 means the null hypothesis is accepted, which contradicts the definition of how p-values are used in hypothesis testing.\nSCORE: 0/1\n\n## CRITERION: States that the null hypothesis is rejected\nEXPLANATION: Similar to the previous point, the student incorrectly states that a p-value of 0.03 means the alternative hypothesis is rejected and the null is accepted. In reality, a p-value of less than 0.05 would reject the null hypothesis, not accept it.\nSCORE: 0/1\n\n## TOTAL_SCORE: 0/2"}, {'id': 1, 'evaluation': '## CRITERION: States that p-value is less than 0.05\nEXPLANATION: The student\'s response explicitly states "p is less than alpha," which implies a p-value less than 0.05, meeting the criterion.\nSCORE: 1/1\n\n## CRITERION: States that the null hypothesis is rejected\nEXPLANATION: The student directly states that since the con

Sorry, this is a mess. But now we see that the first student response obtained a score of 0/2, while the second student obtained 2/2.